# 02 Advanced Prompting Techniques & Reasoning Paradigms

**Real-World Scenario**: Automated Financial Audit & Math Reasoning Pipeline.

This notebook implements 14 prompting techniques, including Chain-of-Thought (CoT), Few-Shot CoT, **Self-Consistency Voting**, **Tree-of-Thoughts (ToT) Branch Evaluators**, and **Skeleton-of-Thought (SoT) Parallel Generators**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))


## Step 2: Few-Shot Chain-of-Thought (CoT) Prompting

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI

example_prompt = ChatPromptTemplate.from_messages([("user", "{input}"), ("assistant", "{output}")])
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=[
        {
            "input": "A company has 3 factories producing 100 widgets/day. 50 widgets are defective. How many good widgets in 5 days?",
            "output": "1. Daily total production = 3 * 100 = 300 widgets.
2. Daily good widgets = 300 - 50 = 250 widgets.
3. Total good widgets in 5 days = 250 * 5 = 1250 widgets.
Answer: 1250"
        }
    ]
)

cot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the math problem step by step."),
    few_shot_prompt,
    ("user", "{input}")
])

if os.getenv("OPENAI_API_KEY"):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    chain = cot_prompt | llm
    res = chain.invoke({"input": "A store orders 4 crates of 50 apples. 20 apples spoil. How many good apples remain?"})
    print("=== Few-Shot CoT Output ===")
    print(res.content)


## Step 3: Self-Consistency Majority Voting Implementation

In [ ]:
import re
from collections import Counter
from langchain_openai import ChatOpenAI

def run_self_consistency(query: str, num_samples: int = 5):
    print(f"=== Running Self-Consistency Voting (N={num_samples} paths) ===")
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not present.")
        return
        
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7) # Temperature > 0 for diverse paths
    answers = []
    
    for i in range(num_samples):
        res = llm.invoke(f"Solve step by step and end with 'Answer: X': {query}")
        # Extract numerical answer
        match = re.search(r"Answer:\s*(\d+)", res.content)
        ans = match.group(1) if match else "Unknown"
        answers.append(ans)
        print(f"Path {i+1} Output Answer: {ans}")
        
    # Majority Vote Computation
    vote_counts = Counter(answers)
    winning_answer, votes = vote_counts.most_common(1)[0]
    print(f"
--- Majority Vote Result ---")
    print(f"Winning Answer: {winning_answer} ({votes}/{num_samples} votes)")

run_self_consistency("A factory makes 40 cars per day for 6 days. 30 cars are damaged. How many good cars?")


## Step 4: Tree-of-Thoughts (ToT) Branch Evaluator Implementation

In [ ]:
def evaluate_thought_branch(problem: str, thought_branch: str) -> float:
    """Uses an LLM evaluator to score a reasoning thought branch V(s) in [0, 1]."""
    if not os.getenv("OPENAI_API_KEY"): return 0.8
    evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    prompt = f"Problem: {problem}
Proposed Thought Step: {thought_branch}
Rate viability from 0.0 to 1.0. Output float ONLY:"
    res = evaluator_llm.invoke(prompt)
    try:
        return float(res.content.strip())
    except:
        return 0.5

problem = "Maximize throughput for a 3-stage assembly pipeline under 100ms latency constraint."
branches = [
    "Branch A: Increase batch size to 256 to saturate GPU.",
    "Branch B: Apply TensorRT quantization to FP8 and keep batch size 1.",
    "Branch C: Remove model layers randomly."
]

print("=== Tree-of-Thoughts (ToT) Branch Evaluation Scores ===")
for b in branches:
    score = evaluate_thought_branch(problem, b)
    print(f"Score: {score:.2f} | {b}")
